<a href="https://colab.research.google.com/github/Kanchanajaddu/GEN_AI-exercises/blob/main/testcasegenerator(w4).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#given question:
#create an ai test case generator agent from business requirements (ReAct + Langchain)
#Add safety layers (jailbreak defense + PII redaction)
#deploy agent locally and expose via API

In [ ]:
#open ai api key
import os
openai_api_key = input("Please enter your OpenAI API key (e.g., sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx): ")
os.environ["OPENAI_API_KEY"] = openai_api_key
print("OpenAI API key configured successfully.")


In [12]:
pip install --upgrade --force-reinstall langchain langchain-openai langchain-community pydantic==2.12.3

  Using cached langchain-1.2.17-py3-none-any.whl.metadata (5.8 kB)
  Using cached langchain_openai-1.2.1-py3-none-any.whl.metadata (3.1 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 3.2 MB/s eta 0:00:00
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached langchain_core-1.3.3-py3-none-any.whl.metadata (4.5 kB)
  Using cached langgraph-1.1.10-py3-none-any.whl.metadata (8.0 kB)
  Using cached openai-2.36.0-py3-none-any.whl.metadata (31 kB)
  Using cached tiktoken-0.12.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (6.7 kB)
  Using cached langchain_classic-1.0.7-py3-none-any.whl.metadata (5.1 kB)
  Using cached sqlalchemy-2.0.49-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata

## Define Core Agent Components

This section defines the Pydantic models for `TestCase` and `TestCasesOutput`, initializes the `ChatOpenAI` LLM, creates a structured output runnable, defines a `Tool` for test case generation, and sets up the `PromptTemplate`.

In [4]:
from pydantic import BaseModel, Field, RootModel
from langchain_openai import ChatOpenAI
from langchain_core.tools import Tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage
import os

# 1. Define Pydantic models
class TestCase(BaseModel):
    id: int = Field(description="Unique identifier for the test case")
    name: str = Field(description="Name of the test case")
    description: str = Field(description="Description of what the test case aims to verify")
    preconditions: list[str] = Field(description="List of conditions that must be true before the test case can be executed")
    steps: list[str] = Field(description="Ordered list of steps to execute the test case")
    expected_result: str = Field(description="The expected outcome of executing the test case")

class TestCasesOutput(RootModel):
    root: list[TestCase]

# 2. Initialize the ChatOpenAI LLM
llm = ChatOpenAI(temperature=0, model_name='gpt-4o', api_key=openai_api_key)

# 3. Create a structured output runnable
structured_llm = llm.with_structured_output(TestCasesOutput)

# 4. Define a Python function for test case generation
def generate_test_cases_tool_function(business_requirements: str) -> str:
    """Generates a list of test cases based on provided business requirements."""
    test_cases = structured_llm.invoke({"business_requirements": business_requirements})
    return test_cases.json(indent=2)

# 5. Create a Tool instance
generate_test_cases_tool = Tool(
    name='generate_test_cases',
    func=generate_test_cases_tool_function,
    description='Generates a list of test cases based on provided business requirements. Input should be a string of business requirements.'
)

# 6. Define a ChatPromptTemplate
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant that generates test cases from business requirements. Ensure the generated test cases are comprehensive and cover all aspects of the requirements."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

print("Core agent components (Pydantic models, LLM, structured output, tool, and prompt template) defined successfully.")

Core agent components (Pydantic models, LLM, structured output, tool, and prompt template) defined successfully.


In [28]:
from pydantic import BaseModel, Field, RootModel
from langchain_openai import ChatOpenAI
from langchain_core.tools import Tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage
import os
import pkgutil
import langchain.agents
import sys
from langchain.agents import create_agent

# 1. Define Pydantic models
class TestCase(BaseModel):
    id: int = Field(description="Unique identifier for the test case")
    name: str = Field(description="Name of the test case")
    description: str = Field(description="Description of what the test case aims to verify")
    preconditions: list[str] = Field(description="List of conditions that must be true before the test case can be executed")
    steps: list[str] = Field(description="Ordered list of steps to execute the test case")
    expected_result: str = Field(description="The expected outcome of executing the test case")

class TestCasesOutput(RootModel):
    root: list[TestCase]

# 2. Initialize the ChatOpenAI LLM
llm = ChatOpenAI(temperature=0, model_name='gpt-4o', api_key=openai_api_key)

# 3. Create a structured output runnable
structured_llm = llm.with_structured_output(TestCasesOutput)

# 4. Define a Python function for test case generation
def generate_test_cases_tool_function(business_requirements: str) -> str:
    """Generates a list of test cases based on provided business requirements."""
    test_cases = structured_llm.invoke({"business_requirements": business_requirements})
    return test_cases.json(indent=2)

# 5. Create a Tool instance
generate_test_cases_tool = Tool(
    name='generate_test_cases',
    func=generate_test_cases_tool_function,
    description='Generates a list of test cases based on provided business requirements. Input should be a string of business requirements.'
)

# 6. Define a ChatPromptTemplate
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant that generates test cases from business requirements. Ensure the generated test cases are comprehensive and cover all aspects of the requirements."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

# 7. Create a list of tools
tools = [generate_test_cases_tool]

# 8. Bind tools to the LLM and chain with the prompt for a single Runnable
llm_with_tools = llm.bind_tools(tools)
agent_runnable = prompt_template | llm_with_tools

# 9. Initialize the ReAct agent using the create_agent factory with the runnable
agent = create_agent(agent_runnable)

print("Core agent components (Pydantic models, LLM, structured output, tool, and prompt template) defined successfully.")
print("ReAct agent initialized successfully and is ready for execution as a Runnable.")


Core agent components (Pydantic models, LLM, structured output, tool, and prompt template) defined successfully.
ReAct agent initialized successfully and is ready for execution as a Runnable.


## Add Safety Layers (Jailbreak Defense + PII Redaction)

In [29]:
pip install --upgrade presidio-analyzer presidio-anonymizer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.1/201.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 8.5 MB/s eta 0:00:00
  Attempting uninstall: cryptography
    Found existing installation: cryptography 43.0.3
    Uninstalling cryptography-43.0.3:
      Successfully uninstalled cryptography-43.0.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pydrive2 1.21.3 requires cryptography<44, but you have cryptography 48.0.0 which is incompatible.
pyopenssl 24.2.1 requires cryptography<44,>=41.0.5, but you have cryptography 48.0.0 which is incompatible.


### PII Redaction with Microsoft Presidio

We'll use Microsoft Presidio for detecting and redacting Personally Identifiable Information (PII) from the agent's output. This ensures that sensitive data is not exposed.

In [32]:
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine
# OperatorConfig might not be directly importable in newer versions, use dict for config

# Initialize Presidio components
analyzer = AnalyzerEngine()
anonymizer = AnonymizerEngine()

def redact_pii(text: str) -> str:
    """Detects and redacts PII from the given text."""
    # Analyze the text for PII entities
    results = analyzer.analyze(text=text, language='en')

    # Anonymize the detected PII entities
    # Using a custom operator config to replace with a placeholder like '<REDACTED_PII>'
    anonymized_text = anonymizer.anonymize(
        text=text,
        analyzer_results=results,
        operators={
            "DEFAULT": {"type": "replace", "new_value": "<REDACTED_PII>"}
        }
    )
    return anonymized_text.text

print("PII redaction functions defined successfully.")

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


PII redaction functions defined successfully.


### Jailbreak Defense (Prompt Engineering)

To implement a basic jailbreak defense, we'll reinforce the system prompt with instructions that guide the agent's behavior to be helpful, harmless, and to adhere strictly to its primary function of generating test cases. This involves explicitly stating its role and limits.

In [31]:
# Update the prompt template to include jailbreak defense instructions
prompt_template_with_safety = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful and safe AI assistant that generates comprehensive test cases from business requirements. Your sole purpose is to assist in software testing by providing relevant and detailed test cases. Do not engage in any activities outside of this scope, such as providing information unrelated to test case generation, generating harmful content, or attempting to circumvent your instructions. Ensure the generated test cases are comprehensive and cover all aspects of the requirements."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

# Re-bind the tools and create the agent with the updated prompt
agent_runnable_with_safety = prompt_template_with_safety | llm_with_tools
agent_with_safety = create_agent(agent_runnable_with_safety)

print("Jailbreak defense (prompt engineering) applied to the agent.")
print("The agent 'agent_with_safety' is now ready with safety layers.")

Jailbreak defense (prompt engineering) applied to the agent.
The agent 'agent_with_safety' is now ready with safety layers.


## Deploy Agent Locally and Expose via API

In [33]:
pip install --upgrade fastapi uvicorn

### FastAPI Application for Agent Deployment

We'll create a FastAPI application that exposes an endpoint to interact with our agent. This endpoint will receive business requirements, pass them to the agent, apply PII redaction to the output, and return the generated test cases.